# Experiment 51 - CatBoost Signal Recovery

## Goal

This experiment completely changes direction from the previous Exp51 leakage attack. The goal is to test whether CatBoost can recover signal that the strongest XGBoost experiments are missing.

The model uses the original categorical variables directly instead of target encoding them. A small set of deterministic interaction and ratio features is added. Three genuinely different CatBoost configurations are evaluated with 5-fold OOF predictions.

After the individual models, in-memory probability and rank blends are tested. 

In [1]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings('ignore')

SEED = 51
N_SPLITS = 5
TARGET = 'Will_Buy_EV'
ID_COL = 'id'

ROOT = Path.cwd()
if not (ROOT / 'data' / 'train.csv').exists():
    ROOT = ROOT.parent

TRAIN_PATH = ROOT / 'data' / 'train.csv'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(TRAIN_PATH)
print('Train shape:', train.shape)

raw_y = train[TARGET]
if pd.api.types.is_numeric_dtype(raw_y):
    y = raw_y.astype(int).to_numpy()
else:
    y_text = raw_y.astype(str).str.strip().str.lower()
    mapping = {'yes': 1, 'no': 0, 'true': 1, 'false': 0, '1': 1, '0': 0}
    y = y_text.map(mapping)
    if y.isna().any():
        raise ValueError(f'Unknown target values: {sorted(y_text[y.isna()].unique())}')
    y = y.astype(int).to_numpy()

print('Target mean:', y.mean())

Train shape: (668665, 15)
Target mean: 0.17464500160768098


In [2]:
NUM_COLS = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level',
]

CAT_COLS = [
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level',
]

def make_features(df):
    out = pd.DataFrame(index=df.index)

    for c in NUM_COLS:
        out[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    for c in CAT_COLS:
        out[c] = df[c].astype(str).fillna('__NA__')

    # Simple deterministic numeric features.
    income = out['Annual_Income_USD'].astype(float)
    age = out['Age'].astype(float)
    commute = out['Daily_Commute_km'].astype(float)
    cars = out['Number_of_Cars_Owned'].astype(float)
    home = out['Charging_Stations_Near_Home'].astype(float)
    work = out['Charging_Stations_Near_Work'].astype(float)
    concern = out['Environmental_Concern_Level'].astype(float)

    out['Income_per_Age'] = income / (age + 1.0)
    out['Income_per_Commute'] = income / (commute + 1.0)
    out['Commute_per_Age'] = commute / (age + 1.0)
    out['Income_per_Car'] = income / (cars + 1.0)
    out['Income_per_Home_Station'] = income / (home + 1.0)
    out['Income_per_Work_Station'] = income / (work + 1.0)
    out['Total_Charging_Stations'] = home + work
    out['Station_Gap'] = home - work
    out['Commute_x_Concern'] = commute * concern
    out['Age_x_Concern'] = age * concern
    out['Income_x_Concern'] = income * concern

    # Low-cardinality categorical interactions for CatBoost.
    out['City_Car'] = out['City_Type'].astype(str) + '__' + out['Current_Car_Type'].astype(str)
    out['City_Subsidy'] = out['City_Type'].astype(str) + '__' + out['Subsidy_Available'].astype(str)
    out['City_HomeCharge'] = out['City_Type'].astype(str) + '__' + out['Home_Charging_Possible'].astype(str)
    out['Car_Subsidy'] = out['Current_Car_Type'].astype(str) + '__' + out['Subsidy_Available'].astype(str)
    out['Car_HomeCharge'] = out['Current_Car_Type'].astype(str) + '__' + out['Home_Charging_Possible'].astype(str)
    out['Anxiety_Subsidy'] = out['Range_Anxiety_Level'].astype(str) + '__' + out['Subsidy_Available'].astype(str)
    out['Anxiety_HomeCharge'] = out['Range_Anxiety_Level'].astype(str) + '__' + out['Home_Charging_Possible'].astype(str)

    # Stable coarse bins.
    out['Income_Bin_5k'] = np.floor(income / 5000.0).astype('float32')
    out['Income_Bin_10k'] = np.floor(income / 10000.0).astype('float32')
    out['Commute_Bin_2'] = np.floor(commute / 2.0).astype('float32')
    out['Commute_Bin_5'] = np.floor(commute / 5.0).astype('float32')
    out['Age_Bin_2'] = np.floor(age / 2.0).astype('float32')
    out['Age_Bin_5'] = np.floor(age / 5.0).astype('float32')

    out = out.replace([np.inf, -np.inf], np.nan)

    explicit_cat_features = [
        'Gender',
        'City_Type',
        'Current_Car_Type',
        'Home_Charging_Possible',
        'Subsidy_Available',
        'Range_Anxiety_Level',
        'City_Car',
        'City_Subsidy',
        'City_HomeCharge',
        'Car_Subsidy',
        'Car_HomeCharge',
        'Anxiety_Subsidy',
        'Anxiety_HomeCharge',
    ]

    for c in explicit_cat_features:
        out[c] = out[c].fillna('__NA__').astype(str)

    return out, explicit_cat_features

X, CAT_FEATURES = make_features(train.drop(columns=[TARGET]))
print('Feature shape:', X.shape)
print('Categorical features:', len(CAT_FEATURES))
print(CAT_FEATURES)

Feature shape: (668665, 37)
Categorical features: 13
['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'City_Car', 'City_Subsidy', 'City_HomeCharge', 'Car_Subsidy', 'Car_HomeCharge', 'Anxiety_Subsidy', 'Anxiety_HomeCharge']


## Model configurations

The three CatBoost models deliberately vary depth, regularization, random strength, bagging temperature, and learning rate. The goal is not a hyperparameter grid. The goal is to create complementary model predictions that can potentially add ranking information when blended.

In [3]:
CAT_INDICES = [X.columns.get_loc(c) for c in CAT_FEATURES]
CONFIGS = {
    'cat_A': dict(
        depth=7,
        learning_rate=0.035,
        iterations=2600,
        l2_leaf_reg=5.0,
        random_strength=1.0,
        bagging_temperature=0.5,
    ),
    'cat_B': dict(
        depth=8,
        learning_rate=0.025,
        iterations=3000,
        l2_leaf_reg=8.0,
        random_strength=0.5,
        bagging_temperature=1.0,
    ),
    'cat_C': dict(
        depth=6,
        learning_rate=0.040,
        iterations=2500,
        l2_leaf_reg=4.0,
        random_strength=1.5,
        bagging_temperature=0.25,
    ),
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_store = {}
result_rows = []

for name, params in CONFIGS.items():
    print('=' * 90)
    print(name)
    print('=' * 90)

    oof = np.zeros(len(X), dtype=np.float64)
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr = X.iloc[tr_idx]
        X_va = X.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        for c in CAT_FEATURES:
            X_tr[c] = X_tr[c].fillna('__NA__').astype(str)
            X_va[c] = X_va[c].fillna('__NA__').astype(str)

        model = CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='AUC',
            verbose=False,
            random_seed=SEED + fold,
            thread_count=-1,
            allow_writing_files=False,
            bootstrap_type='Bayesian',
            **params,
        )

        model.fit(
        X_tr,
        y_tr,
        cat_features=CAT_INDICES,
        eval_set=(X_va, y_va),
        use_best_model=True,
        early_stopping_rounds=180,
        )

        pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = pred
        score = roc_auc_score(y_va, pred)
        fold_scores.append(score)
        print(f'Fold {fold}: {score:.6f} | best iteration: {model.get_best_iteration()}')

        del model, X_tr, X_va, y_tr, y_va, pred
        gc.collect()

    auc = roc_auc_score(y, oof)
    print(f'{name} | OOF ROC-AUC: {auc:.6f}')
    print(f'Fold mean: {np.mean(fold_scores):.6f}')

    oof_store[name] = oof
    result_rows.append({
        'candidate': name,
        'type': 'single',
        'oof_auc': auc,
        'fold_mean': np.mean(fold_scores),
        'fold_std': np.std(fold_scores),
    })

results = pd.DataFrame(result_rows).sort_values('oof_auc', ascending=False)
results

cat_A
Fold 1: 0.940740 | best iteration: 2346
Fold 2: 0.941188 | best iteration: 1853
Fold 3: 0.942608 | best iteration: 1684


KeyboardInterrupt: 

In [ ]:
from scipy.stats import rankdata

def rank01(x):
    return rankdata(x, method='average') / len(x)

blend_rows = []
names = list(oof_store.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a = names[i]
        b = names[j]
        pa = oof_store[a]
        pb = oof_store[b]

        ra = rank01(pa)
        rb = rank01(pb)

        for w in np.arange(0.10, 0.91, 0.05):
            pred_prob = w * pa + (1.0 - w) * pb
            auc_prob = roc_auc_score(y, pred_prob)
            blend_rows.append({
                'candidate': f'prob_{a}_{b}_{w:.2f}',
                'type': 'probability_blend',
                'oof_auc': auc_prob,
                'weight_a': float(w),
            })

            pred_rank = w * ra + (1.0 - w) * rb
            auc_rank = roc_auc_score(y, pred_rank)
            blend_rows.append({
                'candidate': f'rank_{a}_{b}_{w:.2f}',
                'type': 'rank_blend',
                'oof_auc': auc_rank,
                'weight_a': float(w),
            })

blend_results = pd.DataFrame(blend_rows).sort_values('oof_auc', ascending=False)
print('Top blends:')
display(blend_results.head(20))

In [ ]:
all_results = pd.concat([results, blend_results], ignore_index=True, sort=False)
all_results = all_results.sort_values('oof_auc', ascending=False).reset_index(drop=True)

print('BEST EXP51 RESULTS')
display(all_results.head(20))

best = all_results.iloc[0].to_dict()
best_auc = float(best['oof_auc'])

print(f'Best Exp51 OOF ROC-AUC: {best_auc:.6f}')
print('Exp43 reference: 0.945761')
print('Exp41D reference: 0.945724')
print('Exp23B reference: 0.945243')
print(f'Delta vs Exp43: {best_auc - 0.945761:+.6f}')

summary = {
    'experiment': 'Exp51',
    'best_candidate': str(best['candidate']),
    'best_oof_auc': best_auc,
    'exp43_reference': 0.945761,
    'delta_vs_exp43': best_auc - 0.945761,
    'submission_created': False,
    'notes': 'CatBoost raw categorical signal recovery with deterministic interactions and in-memory blends only.'
}

with open(RESULTS_DIR / 'exp51_best_recipe.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

all_results.to_csv(RESULTS_DIR / 'exp51_search_results.csv', index=False)
print('Saved small Exp51 result files.')